# Comparación: Modelo BILP Exacto vs Heurística
Este notebook ejecuta el **modelo matemático completo** (BILP con
$\Phi_{\text{nivel}}$ y $\Phi_{\text{adj}}$) y lo compara contra
la heurística de sustitución + llenado.
**Advertencia**: el ILP con co-ocurrencia escala como $O(n^2 \cdot |S|)$
en variables y restricciones. Para problemas con $>50$ productos la
ejecución puede tomar **horas**. El script incluye estimaciones de
tiempo antes de cada corrida.
### Estructura
| Fase | Contenido | Tiempo estimado |
|---|---|---|
| 1 | Carga de datos y reglas | ~10 s |
| 2 | Benchmark de escalamiento (15–35 productos) | ~5 min |
| 3 | Comparación a tamaño medio (50 productos) | ~30 min |
| 4 | Comparación a tamaño completo (92 productos) | **horas** |
| 5 | Análisis de sensibilidad ($\lambda_1, \lambda_2$) | ~10 min |
| 6 | Tabla de resultados y conclusiones | instantáneo |

## Fase 1: Carga de Datos

In [ ]:
import os, sys, time, random, warnings, datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
random.seed(42)
np.random.seed(42)

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from exploracion import load_planograms, Planogram, Shelf, ProductPlacement
from planogram_model import (
    build_product_catalog, find_best_match, adapt_planogram,
    mine_placement_rules, evaluate_rule_adherence, ProductInfo,
)
import pulp

OUTPUT_DIR = "test_output/ilp_comparison"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def log(msg):
    """Timestamped log."""
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}] {msg}")

log("Iniciando carga de datos...")
t0_global = time.perf_counter()

CSV_ORIGINAL = "datos/ejemplo_planograma.csv"
CSV_NEW = "datos/Ejemplo 2.csv"

planograms_orig = load_planograms(CSV_ORIGINAL)
planograms_new = load_planograms(CSV_NEW)
planograms = planograms_orig + planograms_new

catalog = build_product_catalog([CSV_ORIGINAL, CSV_NEW])

log(f"  |H| = {len(planograms)} planogramas")
log(f"  |P| = {len(catalog)} productos")

# Mine rules for CF (the furniture type of the test planogram)
cf_plans = [p for p in planograms if p.mueble_id == 'CF']
bco_plans = [p for p in cf_plans if p.segmento_id == 'BCO']
rules = mine_placement_rules(bco_plans, catalog)
log(f"  Reglas BCO/CF: {len(rules['level_probs'])} productos, "
    f"{len(rules['adjacencies'])} pares adj.")

# Use smallest planogram as the test template
template = min(planograms_orig,
               key=lambda p: len({pr.upc for s in p.shelves.values()
                                  for pr in s.products}))
log(f"  Template: {template.title}")

# Extract template info
S_items = list(template.shelves.items())
S_idx = list(range(len(S_items)))
S_arr = [s for _, s in S_items]
n_shelves = len(S_idx)

# Effective shelf heights from Y gaps
ys = sorted({s.y for s in S_arr})
eff_h = {}
for i, yv in enumerate(ys):
    if i < len(ys) - 1:
        eff_h[yv] = ys[i + 1] - yv
    else:
        eff_h[yv] = (ys[i] - ys[i - 1]) if i > 0 else 40.0
for shelf in S_arr:
    shelf.shelf_height = eff_h[shelf.y]

# Full product set from template
all_upcs = sorted({pr.upc for s in template.shelves.values()
                   for pr in s.products})
all_upcs = [u for u in all_upcs if u in catalog
            and any(catalog[u].height <= s.shelf_height * 1.05 for s in S_arr)]

tfac = {p.upc: p.facings for s in template.shelves.values()
        for p in s.products}

log(f"  Productos factibles: {len(all_upcs)}")
log(f"  Charolas: {n_shelves}")

t_load = time.perf_counter() - t0_global
log(f"  Carga completada en {t_load:.1f}s")

## Fase 2: Benchmark de Escalamiento
Ejecutamos el ILP completo (con co-ocurrencia, $\lambda_2 > 0$) en
subconjuntos de 15, 20, 25, 30 y 35 productos para medir cómo escala
el tiempo de solución. Con estos datos estimamos el tiempo para
problemas más grandes.

In [ ]:
# =============================================================
# FUNCIÓN: RESOLVER ILP COMPLETO
# =============================================================
def solve_ilp(P_set, S_items, S_idx, S_arr, catalog, tfac,
              rules, lambda1=1.0, lambda2=0.5,
              time_limit=3600, gap_rel=0.01, verbose=False):
    """
    Resuelve el BILP completo (Φ_nivel + Φ_adj) para un conjunto
    de productos P_set asignados a charolas S.

    Retorna un dict con:
      - status, objective, phi_nivel, phi_adj
      - assignment: {upc: shelf_index}
      - time, n_vars, n_constraints
    """
    lp = rules['level_probs']
    ac = rules['adjacencies']
    N_H = rules['n_planograms']
    n_P = len(P_set)

    # Pares para co-ocurrencia
    pairs = []
    if lambda2 > 0:
        for i in range(n_P):
            for k in range(i + 1, n_P):
                pairs.append((P_set[i], P_set[k]))

    prob = pulp.LpProblem("Planogram_BILP", pulp.LpMaximize)

    # Variables x_ps
    x = {}
    for pi, p in enumerate(P_set):
        for si in S_idx:
            x[p, si] = pulp.LpVariable(f"x_{pi}_{si}", cat="Binary")

    # Variables y_pqs (co-ubicación)
    y = {}
    if lambda2 > 0 and pairs:
        for pair_i, (p, q) in enumerate(pairs):
            for si in S_idx:
                y[p, q, si] = pulp.LpVariable(f"y_{pair_i}_{si}", cat="Binary")

    # Objetivo: λ1·Φ_nivel + λ2·Φ_adj
    phi_nivel_expr = (1.0 / max(n_P, 1)) * pulp.lpSum(
        lp.get(p, {}).get(S_arr[si].level, 0.0) * x[p, si]
        for p in P_set for si in S_idx)

    if lambda2 > 0 and pairs:
        n_pairs = len(pairs)
        phi_adj_expr = (1.0 / max(n_pairs, 1)) * pulp.lpSum(
            (ac.get(tuple(sorted([p, q])), 0) / max(N_H, 1)) * y[p, q, si]
            for p, q in pairs for si in S_idx)
    else:
        phi_adj_expr = 0

    prob += lambda1 * phi_nivel_expr + lambda2 * phi_adj_expr, "FO"

    # (6) Cobertura: cada producto en exactamente un anaquel
    for p in P_set:
        prob += pulp.lpSum(x[p, si] for si in S_idx) == 1, f"cov_{p}"

    # (7) Capacidad de ancho
    for si, shelf in zip(S_idx, S_arr):
        prob += pulp.lpSum(
            catalog[p].width * tfac.get(p, 1) * x[p, si]
            for p in P_set) <= shelf.shelf_width, f"w_{si}"

    # (8) Compatibilidad de altura
    for p in P_set:
        h_p = catalog[p].height
        for si, shelf in zip(S_idx, S_arr):
            if h_p > shelf.shelf_height * 1.05:
                prob += x[p, si] == 0, f"ht_{p}_{si}"

    # (9)-(11) McCormick: y_pqs = x_ps · x_qs
    if lambda2 > 0 and pairs:
        for pair_i, (p, q) in enumerate(pairs):
            for si in S_idx:
                prob += y[p, q, si] <= x[p, si], f"mc1_{pair_i}_{si}"
                prob += y[p, q, si] <= x[q, si], f"mc2_{pair_i}_{si}"
                prob += y[p, q, si] >= x[p, si] + x[q, si] - 1, f"mc3_{pair_i}_{si}"

    # Contar variables y restricciones
    n_x = len(x)
    n_y = len(y)
    n_vars = n_x + n_y
    n_cons = len(prob.constraints)

    # Resolver
    solver = pulp.PULP_CBC_CMD(timeLimit=time_limit, msg=int(verbose),
                                gapRel=gap_rel)
    t0 = time.perf_counter()
    status = prob.solve(solver)
    solve_time = time.perf_counter() - t0

    # Extraer asignación
    assignment = {}
    if prob.status == 1:
        for p in P_set:
            for si in S_idx:
                v = pulp.value(x[p, si])
                if v is not None and v > 0.5:
                    assignment[p] = si
                    break

    # Calcular Φ por separado
    obj_val = pulp.value(prob.objective)
    phi_n = pulp.value(phi_nivel_expr) if prob.status == 1 else None
    phi_a = pulp.value(phi_adj_expr) if prob.status == 1 and lambda2 > 0 else None

    return {
        'status': pulp.LpStatus[status],
        'objective': obj_val,
        'phi_nivel': phi_n,
        'phi_adj': phi_a,
        'assignment': assignment,
        'time': solve_time,
        'n_x': n_x,
        'n_y': n_y,
        'n_vars': n_vars,
        'n_constraints': n_cons,
        'n_products': n_P,
        'lambda1': lambda1,
        'lambda2': lambda2,
        'gap_rel': gap_rel,
    }


# =============================================================
# FUNCIÓN: EVALUAR HEURÍSTICA CON MISMAS MÉTRICAS
# =============================================================
def eval_heuristic(P_set, template, planograms, catalog, rules):
    """
    Ejecuta la heurística y calcula Φ_nivel y Φ_adj usando las
    mismas fórmulas que el ILP para comparación directa.
    """
    lp = rules['level_probs']
    ac = rules['adjacencies']
    N_H = rules['n_planograms']

    target_set = set(P_set)
    t0 = time.perf_counter()
    match, overlap, total = find_best_match(
        target_set, template.tamano, planograms,
        target_mueble=template.mueble_id,
    )
    gen = adapt_planogram(match, target_set, catalog)
    h_time = time.perf_counter() - t0

    # Calcular Φ_nivel
    placed = []
    for s in gen.shelves.values():
        for p in s.products:
            placed.append((p.upc, s.level))

    n_placed = len(placed)
    phi_nivel = sum(lp.get(u, {}).get(lv, 0.0) for u, lv in placed) / max(n_placed, 1)

    # Calcular Φ_adj
    n_pairs_total = 0
    adj_sum = 0
    for s in gen.shelves.values():
        upcs = [p.upc for p in s.products]
        for i in range(len(upcs)):
            for j in range(i + 1, len(upcs)):
                pair = tuple(sorted([upcs[i], upcs[j]]))
                adj_sum += ac.get(pair, 0) / max(N_H, 1)
                n_pairs_total += 1
    phi_adj = adj_sum / max(n_pairs_total, 1) if n_pairs_total > 0 else 0

    return {
        'phi_nivel': phi_nivel,
        'phi_adj': phi_adj,
        'n_placed': n_placed,
        'time': h_time,
        'planogram': gen,
    }



In [ ]:
# =============================================================
# BENCHMARK DE ESCALAMIENTO
# =============================================================
log("=" * 70)
log("FASE 2: BENCHMARK DE ESCALAMIENTO")
log("=" * 70)

benchmark_sizes = [15, 20, 25, 30, 35]
benchmark_results = []

for n in benchmark_sizes:
    if n > len(all_upcs):
        break

    # Subconjunto aleatorio
    random.seed(42)
    P_sub = sorted(random.sample(all_upcs, n))

    # Estimar tamaño del problema
    n_pairs = n * (n - 1) // 2
    n_y_est = n_pairs * n_shelves
    n_mc_est = 3 * n_y_est
    n_vars_est = n * n_shelves + n_y_est
    n_cons_est = n + n_shelves + n * n_shelves + n_mc_est

    log(f"\n--- n={n} productos ---")
    log(f"  Variables estimadas: {n_vars_est:,} "
        f"(x={n*n_shelves}, y={n_y_est:,})")
    log(f"  Restricciones estimadas: {n_cons_est:,}")

    # ILP con λ2=0.5
    log(f"  Resolviendo ILP (λ1=1, λ2=0.5, timeLimit=600s)...")
    t0 = time.perf_counter()
    ilp = solve_ilp(P_sub, S_items, S_idx, S_arr, catalog, tfac,
                    rules, lambda1=1.0, lambda2=0.5,
                    time_limit=600, gap_rel=0.01)
    log(f"  ILP: {ilp['status']}, obj={ilp['objective']:.4f}, "
        f"Φ_nivel={ilp['phi_nivel']:.4f}, Φ_adj={ilp['phi_adj']:.4f}, "
        f"t={ilp['time']:.2f}s")

    # Heurística
    h = eval_heuristic(P_sub, template, planograms, catalog, rules)
    log(f"  Heurística: Φ_nivel={h['phi_nivel']:.4f}, "
        f"Φ_adj={h['phi_adj']:.4f}, colocados={h['n_placed']}/{n}, "
        f"t={h['time']*1000:.1f}ms")

    benchmark_results.append({
        'n': n,
        'ilp_time': ilp['time'],
        'ilp_phi_nivel': ilp['phi_nivel'],
        'ilp_phi_adj': ilp['phi_adj'],
        'ilp_obj': ilp['objective'],
        'ilp_status': ilp['status'],
        'ilp_n_vars': ilp['n_vars'],
        'ilp_n_cons': ilp['n_constraints'],
        'h_time': h['time'],
        'h_phi_nivel': h['phi_nivel'],
        'h_phi_adj': h['phi_adj'],
        'h_n_placed': h['n_placed'],
    })

# Tabla de benchmark
print("\n")
log("TABLA DE ESCALAMIENTO")
print(f"{'n':>4} {'Vars':>10} {'Cons':>10} {'ILP_t(s)':>10} "
      f"{'ILP_Φniv':>10} {'ILP_Φadj':>10} {'H_Φniv':>10} {'H_Φadj':>10} "
      f"{'Gap_niv':>10} {'Status':>10}")
print("-" * 105)
for r in benchmark_results:
    gap = ((r['ilp_phi_nivel'] or 0) - r['h_phi_nivel']) / max(r['ilp_phi_nivel'] or 1, 0.001) * 100
    print(f"{r['n']:>4} {r['ilp_n_vars']:>10,} {r['ilp_n_cons']:>10,} "
          f"{r['ilp_time']:>10.2f} {r['ilp_phi_nivel'] or 0:>10.4f} "
          f"{r['ilp_phi_adj'] or 0:>10.4f} {r['h_phi_nivel']:>10.4f} "
          f"{r['h_phi_adj']:>10.4f} {gap:>9.1f}% {r['ilp_status']:>10}")

# Estimación de tiempo para problemas grandes
if len(benchmark_results) >= 3:
    sizes = [r['n'] for r in benchmark_results]
    times = [r['ilp_time'] for r in benchmark_results]

    # Fit exponential: log(t) = a + b*n
    from numpy.polynomial import polynomial as P
    log_times = np.log(np.array(times) + 1e-6)
    coeffs = np.polyfit(sizes, log_times, 1)

    print("\n")
    log("ESTIMACIÓN DE TIEMPOS (extrapolación exponencial)")
    for n_est in [40, 50, 60, 70, 80, 92]:
        t_est = np.exp(coeffs[1] + coeffs[0] * n_est)
        if t_est < 60:
            t_str = f"{t_est:.0f}s"
        elif t_est < 3600:
            t_str = f"{t_est/60:.0f}min"
        else:
            t_str = f"{t_est/3600:.1f}h"
        log(f"  n={n_est}: ~{t_str} estimado")

## Fase 3: Comparación a Tamaño Medio (50 productos)
Con 50 productos y $\lambda_2=0.5$:
- Variables $y$: $\binom{50}{2} \times 18 = 22{,}050$
- Restricciones McCormick: $3 \times 22{,}050 = 66{,}150$
Time limit: 1 hora.

In [ ]:
log("\n" + "=" * 70)
log("FASE 3: COMPARACIÓN TAMAÑO MEDIO (50 productos)")
log("=" * 70)

N_MED = min(50, len(all_upcs))
random.seed(42)
P_med = sorted(random.sample(all_upcs, N_MED))

n_pairs_med = N_MED * (N_MED - 1) // 2
log(f"  Productos: {N_MED}")
log(f"  Pares co-ocurrencia: {n_pairs_med:,}")
log(f"  Variables y estimadas: {n_pairs_med * n_shelves:,}")
log(f"  Time limit: 3600s (1 hora)")

# ILP completo
log(f"\n  Resolviendo ILP (λ1=1, λ2=0.5)...")
ilp_med = solve_ilp(P_med, S_items, S_idx, S_arr, catalog, tfac,
                    rules, lambda1=1.0, lambda2=0.5,
                    time_limit=3600, gap_rel=0.02)
log(f"  Status: {ilp_med['status']}")
log(f"  Objetivo: {ilp_med['objective']:.6f}")
log(f"  Φ_nivel: {ilp_med['phi_nivel']:.4f}")
log(f"  Φ_adj: {ilp_med['phi_adj']:.4f}")
log(f"  Tiempo: {ilp_med['time']:.1f}s ({ilp_med['time']/60:.1f} min)")
log(f"  Variables: {ilp_med['n_vars']:,}")
log(f"  Restricciones: {ilp_med['n_constraints']:,}")

# ILP sin co-ocurrencia (referencia rápida)
log(f"\n  Resolviendo ILP (λ1=1, λ2=0, solo Φ_nivel)...")
ilp_med_l0 = solve_ilp(P_med, S_items, S_idx, S_arr, catalog, tfac,
                        rules, lambda1=1.0, lambda2=0.0,
                        time_limit=120, gap_rel=0.01)
log(f"  Φ_nivel (solo): {ilp_med_l0['phi_nivel']:.4f}, t={ilp_med_l0['time']:.1f}s")

# Heurística
h_med = eval_heuristic(P_med, template, planograms, catalog, rules)
log(f"\n  Heurística:")
log(f"  Φ_nivel: {h_med['phi_nivel']:.4f}")
log(f"  Φ_adj: {h_med['phi_adj']:.4f}")
log(f"  Colocados: {h_med['n_placed']}/{N_MED}")
log(f"  Tiempo: {h_med['time']*1000:.1f}ms")

# Gaps
gap_n = ((ilp_med['phi_nivel'] or 0) - h_med['phi_nivel']) / max(ilp_med['phi_nivel'] or 1, 0.001) * 100
gap_a = ((ilp_med['phi_adj'] or 0) - h_med['phi_adj']) / max(ilp_med['phi_adj'] or 1, 0.001) * 100
log(f"\n  Gap Φ_nivel: {gap_n:+.1f}%")
log(f"  Gap Φ_adj: {gap_a:+.1f}%")
log(f"  Factor de velocidad: {ilp_med['time'] / max(h_med['time'], 1e-6):.0f}×")

## Fase 4: Comparación a Tamaño Completo (92 productos)
Con 92 productos y $\lambda_2=0.5$:
- Variables $y$: $\binom{92}{2} \times 18 = 75{,}348$
- Restricciones McCormick: $3 \times 75{,}348 = 226{,}044$
**Time limit: 4 horas.** El solver puede no alcanzar el óptimo global,
pero reportará la mejor cota inferior (*incumbent*) y la cota superior
(*best bound*), permitiendo calcular el gap de optimalidad.

In [ ]:
log("\n" + "=" * 70)
log("FASE 4: COMPARACIÓN TAMAÑO COMPLETO (92 productos)")
log("=" * 70)

N_FULL = len(all_upcs)
P_full = all_upcs

n_pairs_full = N_FULL * (N_FULL - 1) // 2
log(f"  Productos: {N_FULL}")
log(f"  Pares co-ocurrencia: {n_pairs_full:,}")
log(f"  Variables y estimadas: {n_pairs_full * n_shelves:,}")
log(f"  Time limit: 14400s (4 horas)")
log(f"  INICIO: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ILP completo (4h time limit)
log(f"\n  Resolviendo ILP (λ1=1, λ2=0.5)...")
log(f"  Esto puede tomar HORAS. Verificar estimaciones de Fase 2.")
ilp_full = solve_ilp(P_full, S_items, S_idx, S_arr, catalog, tfac,
                     rules, lambda1=1.0, lambda2=0.5,
                     time_limit=14400, gap_rel=0.05)
log(f"  Status: {ilp_full['status']}")
log(f"  Objetivo: {ilp_full['objective']:.6f}" if ilp_full['objective'] else "  Sin solución factible")
log(f"  Φ_nivel: {ilp_full['phi_nivel']:.4f}" if ilp_full['phi_nivel'] else "  Φ_nivel: N/A")
log(f"  Φ_adj: {ilp_full['phi_adj']:.4f}" if ilp_full['phi_adj'] else "  Φ_adj: N/A")
log(f"  Tiempo: {ilp_full['time']:.1f}s ({ilp_full['time']/3600:.2f}h)")
log(f"  Variables: {ilp_full['n_vars']:,}")
log(f"  Restricciones: {ilp_full['n_constraints']:,}")

# ILP solo Φ_nivel (referencia, ya sabemos que toma ~28s)
log(f"\n  Resolviendo ILP (λ1=1, λ2=0)...")
ilp_full_l0 = solve_ilp(P_full, S_items, S_idx, S_arr, catalog, tfac,
                         rules, lambda1=1.0, lambda2=0.0,
                         time_limit=300, gap_rel=0.01)
log(f"  Φ_nivel (solo): {ilp_full_l0['phi_nivel']:.4f}, t={ilp_full_l0['time']:.1f}s")

# Heurística
h_full = eval_heuristic(P_full, template, planograms, catalog, rules)
log(f"\n  Heurística:")
log(f"  Φ_nivel: {h_full['phi_nivel']:.4f}")
log(f"  Φ_adj: {h_full['phi_adj']:.4f}")
log(f"  Colocados: {h_full['n_placed']}/{N_FULL}")
log(f"  Tiempo: {h_full['time']*1000:.1f}ms")

# Gaps
if ilp_full['phi_nivel']:
    gap_n = (ilp_full['phi_nivel'] - h_full['phi_nivel']) / max(ilp_full['phi_nivel'], 0.001) * 100
    log(f"\n  Gap Φ_nivel: {gap_n:+.1f}%")
if ilp_full['phi_adj']:
    gap_a = (ilp_full['phi_adj'] - h_full['phi_adj']) / max(ilp_full['phi_adj'], 0.001) * 100
    log(f"  Gap Φ_adj: {gap_a:+.1f}%")
log(f"  Factor de velocidad: {ilp_full['time'] / max(h_full['time'], 1e-6):.0f}×")
log(f"  FIN: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## Fase 5: Análisis de Sensibilidad ($\lambda$)
Variamos los pesos $\lambda_1$ y $\lambda_2$ en un subconjunto de
25 productos (rápido) para observar el trade-off entre $\Phi_{\text{nivel}}$
y $\Phi_{\text{adj}}$.

In [ ]:
log("\n" + "=" * 70)
log("FASE 5: ANÁLISIS DE SENSIBILIDAD (λ)")
log("=" * 70)

N_SENS = 25
random.seed(42)
P_sens = sorted(random.sample(all_upcs, N_SENS))

lambda_configs = [
    (1.0, 0.0),   # solo nivel
    (1.0, 0.1),
    (1.0, 0.25),
    (1.0, 0.5),
    (1.0, 1.0),
    (1.0, 2.0),
    (0.0, 1.0),   # solo adyacencia
]

sens_results = []

for l1, l2 in lambda_configs:
    log(f"  λ1={l1}, λ2={l2}...")
    r = solve_ilp(P_sens, S_items, S_idx, S_arr, catalog, tfac,
                  rules, lambda1=l1, lambda2=l2,
                  time_limit=300, gap_rel=0.01)
    log(f"    Φ_nivel={r['phi_nivel']:.4f}, Φ_adj={r['phi_adj']:.4f}, "
        f"obj={r['objective']:.4f}, t={r['time']:.1f}s")
    sens_results.append({
        'lambda1': l1, 'lambda2': l2,
        'phi_nivel': r['phi_nivel'],
        'phi_adj': r['phi_adj'],
        'objective': r['objective'],
        'time': r['time'],
        'status': r['status'],
    })

# Heurística (referencia, independiente de lambda)
h_sens = eval_heuristic(P_sens, template, planograms, catalog, rules)
log(f"\n  Heurística (referencia): Φ_nivel={h_sens['phi_nivel']:.4f}, "
    f"Φ_adj={h_sens['phi_adj']:.4f}")

# Tabla
print("\n")
print(f"{'λ1':>4} {'λ2':>4} {'Φ_nivel':>10} {'Φ_adj':>10} "
      f"{'Objetivo':>10} {'Tiempo':>8} {'Status':>10}")
print("-" * 62)
for r in sens_results:
    print(f"{r['lambda1']:>4.1f} {r['lambda2']:>4.1f} "
          f"{r['phi_nivel'] or 0:>10.4f} {r['phi_adj'] or 0:>10.4f} "
          f"{r['objective'] or 0:>10.4f} {r['time']:>7.1f}s {r['status']:>10}")
print(f"{'H':>4} {'---':>4} {h_sens['phi_nivel']:>10.4f} "
      f"{h_sens['phi_adj']:>10.4f} {'---':>10} "
      f"{h_sens['time']*1000:>6.1f}ms {'---':>10}")

# Gráfica trade-off
fig, ax = plt.subplots(figsize=(8, 5))
phi_ns = [r['phi_nivel'] or 0 for r in sens_results]
phi_as = [r['phi_adj'] or 0 for r in sens_results]
labels = [f"({r['lambda1']},{r['lambda2']})" for r in sens_results]

ax.scatter(phi_ns, phi_as, s=100, c='steelblue', zorder=5)
for i, lb in enumerate(labels):
    ax.annotate(lb, (phi_ns[i], phi_as[i]), fontsize=8,
                textcoords="offset points", xytext=(5, 5))

ax.scatter([h_sens['phi_nivel']], [h_sens['phi_adj']],
           s=150, c='crimson', marker='*', zorder=6, label='Heurística')
ax.set_xlabel('Φ_nivel')
ax.set_ylabel('Φ_adj')
ax.set_title('Trade-off Φ_nivel vs Φ_adj (ILP, n=25)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "lambda_tradeoff.png"), dpi=150)
plt.close(fig)
log(f"  Gráfica guardada: {OUTPUT_DIR}/lambda_tradeoff.png")

## Fase 6: Tabla de Resultados y Conclusiones

In [ ]:
# =============================================================
# TABLA RESUMEN FINAL
# =============================================================
log("\n" + "=" * 70)
log("FASE 6: TABLA RESUMEN FINAL")
log("=" * 70)

total_time = time.perf_counter() - t0_global

print(f"\n{'Método':<25} {'n':>4} {'Φ_nivel':>10} {'Φ_adj':>10} "
      f"{'Colocados':>10} {'Tiempo':>12} {'Status':>10}")
print("=" * 90)

# Benchmark rows
for r in benchmark_results:
    t_str = f"{r['ilp_time']:.1f}s" if r['ilp_time'] < 60 else f"{r['ilp_time']/60:.1f}min"
    print(f"{'ILP (λ2=0.5)':<25} {r['n']:>4} {r['ilp_phi_nivel'] or 0:>10.4f} "
          f"{r['ilp_phi_adj'] or 0:>10.4f} {r['n']:>10} {t_str:>12} {r['ilp_status']:>10}")

# Medium
t_str = f"{ilp_med['time']:.1f}s" if ilp_med['time'] < 60 else f"{ilp_med['time']/60:.1f}min"
print(f"{'ILP (λ2=0.5)':<25} {N_MED:>4} {ilp_med['phi_nivel'] or 0:>10.4f} "
      f"{ilp_med['phi_adj'] or 0:>10.4f} {N_MED:>10} {t_str:>12} {ilp_med['status']:>10}")

# Full
if ilp_full['time'] < 3600:
    t_str = f"{ilp_full['time']/60:.1f}min"
else:
    t_str = f"{ilp_full['time']/3600:.1f}h"
print(f"{'ILP (λ2=0.5)':<25} {N_FULL:>4} {ilp_full['phi_nivel'] or 0:>10.4f} "
      f"{ilp_full['phi_adj'] or 0:>10.4f} {N_FULL:>10} {t_str:>12} {ilp_full['status']:>10}")

# ILP lambda2=0
t_str = f"{ilp_full_l0['time']:.1f}s"
print(f"{'ILP (λ2=0)':<25} {N_FULL:>4} {ilp_full_l0['phi_nivel'] or 0:>10.4f} "
      f"{'---':>10} {N_FULL:>10} {t_str:>12} {ilp_full_l0['status']:>10}")

# Heuristic
print(f"{'Heurística':<25} {N_FULL:>4} {h_full['phi_nivel']:>10.4f} "
      f"{h_full['phi_adj']:>10.4f} {h_full['n_placed']:>10} "
      f"{h_full['time']*1000:.1f}ms{' ':>6} {'---':>10}")

print(f"\nTiempo total de ejecución: {total_time:.0f}s ({total_time/3600:.2f}h)")